In [9]:
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib.collections import LineCollection
import matplotlib
from google.colab import files

# 용량 제한 해제
matplotlib.rcParams['animation.embed_limit'] = 500

# 요청하신 색상 코드
# 0: 파랑 (현재 진행 중)
# 1: 회색 (완료/배경/대기)
# 2: 초록 (MWPM 형성)
# 3: 보라 (최종 TSP 경로)
COLOR_MAP = {0: '#3498db', 1: '#e0e0e0', 2: '#2ecc71', 3: '#9b59b6'}

file_list = [
    # 'myown_a280.txt',
    # 'MST2_a280.txt',
    'basic_CH_a280.txt'
]

for filepath in file_list:
    if not os.path.exists(filepath):
        print(f"❌ 파일을 찾을 수 없습니다: {filepath}")
        continue

    print(f"\n▶ [{filepath}] MP4 영상 렌더링 중...")

    frames = []
    stage_names = []
    current_edges = []
    current_stage = ""

    with open(filepath, 'r') as f:
        for line in f:
            line = line.strip()
            if not line: continue
            if line.startswith("FRAME:"):
                if current_edges or current_stage:
                    frames.append(current_edges)
                    stage_names.append(current_stage)
                current_stage = line.split("FRAME:", 1)[1].strip()
                current_edges = []
            else:
                parts = list(map(float, line.split()))
                if len(parts) >= 4:
                    current_edges.append(parts)

    if current_edges or current_stage:
        frames.append(current_edges)
        stage_names.append(current_stage)

    # 캔버스 설정 (축 제거하여 TSP 선에만 집중)
    fig, ax = plt.subplots(figsize=(10, 8))
    ax.axis('off')

    nodes = set()
    for frame in frames:
        for e in frame:
            nodes.add((e[0], e[1]))
            nodes.add((e[2], e[3]))

    if not nodes:
        plt.close(fig)
        continue

    nx = [p[0] for p in nodes]
    ny = [p[1] for p in nodes]
    # 빨간 점들 찍기
    ax.scatter(nx, ny, c='#e74c3c', s=15, zorder=5)

    lc = LineCollection([], linewidths=1.5, zorder=1)
    ax.add_collection(lc)
    title = ax.set_title("")

    def update(i):
        segments = []
        seg_colors = []
        for e in frames[i]:
            segments.append([(e[0], e[1]), (e[2], e[3])])
            # 색상 태그 파싱 (없으면 기본 파랑 0)
            c_code = int(e[4]) if len(e) >= 5 else 0
            seg_colors.append(COLOR_MAP.get(c_code, '#3498db'))

        lc.set_segments(segments)
        lc.set_color(seg_colors)
        title.set_text(f"[{filepath.split('.')[0]}] {stage_names[i]} ({i+1}/{len(frames)})")
        return lc, title

    # MP4 인코딩
    ani = animation.FuncAnimation(fig, update, frames=len(frames), interval=100, blit=False)
    mp4_filename = filepath.replace(".txt", "_animation.mp4")

    # 초당 15프레임으로 약간 부드럽게 설정
    ani.save(mp4_filename, writer='ffmpeg', fps=15)
    plt.close(fig)
    print(f"✅ 저장 완료: {mp4_filename}")

    # 완료 시 즉시 다운로드 창 호출
    try:
        files.download(mp4_filename)
    except:
        pass

print("\n🎉 모든 동영상 저장이 완료되었습니다!")


▶ [basic_CH_a280.txt] MP4 영상 렌더링 중...
✅ 저장 완료: basic_CH_a280_animation.mp4


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


🎉 모든 동영상 저장이 완료되었습니다!
